## Git importation and dependencies installation

In [1]:
!git clone https://github.com/brakoto/LOG6305_tp4.git

Cloning into 'LOG6305_tp4'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 104 (delta 48), reused 78 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 344.18 KiB | 1.65 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [2]:
%cd LOG6305_tp4/

/content/LOG6305_tp4


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 8.5 MB/s eta 0:00:00


In [4]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


## Import depedencies

In [5]:
import importlib
from poly_llm.common.abstract_executor import AbstractExecutor
from poly_llm.common.prompt_generator import PromptGenerator
from poly_llm.generators.llm_test_generator import LLMTestGenerator
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

## Model Loading

In [6]:
from poly_llm.common.prompt_generator import PromptGenerator
from poly_llm.generators.llm_test_generator import LLMTestGenerator
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Salesforce/codegen-6B-mono"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_name,
      quantization_config=bnb_config,
      device_map="auto", trust_remote_code=True,
      dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.3G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.3G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-6B-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...32}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def create_test_function(tested_file, few_shot_examples=[]):
  prompt_generator = PromptGenerator(tested_file)
  llm_generator = LLMTestGenerator(model, tokenizer, tested_file)
  if few_shot_examples:
    prompt = prompt_generator.generate_prompt(few_shot_examples=few_shot_examples)
  else:
    prompt = prompt_generator.generate_prompt()
  print(prompt)
  test, test_name = llm_generator.create_test_function(prompt)
  return test, test_name


In [8]:
def save_test_to_file(test, tested_file,filename="test.py"):
  llm_generator = LLMTestGenerator(model, tokenizer, tested_file)
  llm_generator.write_test_to_file(test, filename=filename)

In [9]:
def coverage_llm_produced_code(filename, tested_function, test_name):
  module_name = filename.split(".")[0]
  function_name = test_name

  module = importlib.import_module(module_name)
  function = getattr(module, function_name)
  executor2 = AbstractExecutor(function)

  coverage_data = executor2._execute_input(tested_function)
  return coverage_data

In [ ]:
def delete_model(model, tokenizer):
  del model
  del tokenizer
  torch.cuda.empty_cache()

## `find_closest_elements` Test generation

In [10]:
from poly_llm.to_test.find_closest_elements import find_closest_elements

### Few-shot Test Generation with the LLM (`find_closest_elements`)

In [11]:
few_shot_examples = ['''def test_find_closest_elements(): \n
    assert find_closest_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2]) == (3.9, 4.0) \n
    assert find_closest_elements([1.0, 2.0, 5.9, 4.0, 5.0]) == (5.0, 5.9) \n''']
few_find_closest_elements_test, few_find_closest_elements_test_name = create_test_function(find_closest_elements, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_find_closest_elements_test_filename = "code_gen_few_test_find_closest_elements.py"
save_test_to_file(few_find_closest_elements_test, find_closest_elements, filename=few_find_closest_elements_test_filename)

Generate unit tests for the provided input file.
Input
def find_closest_elements(numbers: List[float]) -> Tuple[float, float]:
    
    closest_pair = None
    distance = None

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                if distance is None:
                    distance = abs(elem - elem2)
                    closest_pair = tuple(sorted([elem, elem2]))
                else:
                    new_distance = abs(elem - elem2)
                    if new_distance < distance:
                        distance = new_distance
                        closest_pair = tuple(sorted([elem, elem2]))

    return closest_pair

Below is a list of test case examples.
Example
def test_find_closest_elements(): 

    assert find_closest_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2]) == (3.9, 4.0) 

    assert find_closest_elements([1.0, 2.0, 5.9, 4.0, 5.0]) == (5.0, 5.9) 


Test function written to code_gen_few_test_find_clo

### Generated test coverage (`find_closest_elements`)

In [13]:

few_find_closest_elements_llm_test_coverage = coverage_llm_produced_code(few_find_closest_elements_test_filename, find_closest_elements, few_find_closest_elements_test_name)
print(few_find_closest_elements_llm_test_coverage)

few_line_cov = few_find_closest_elements_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_find_closest_elements_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')


print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

{'input': <function find_closest_elements at 0x79268cc847c0>, 'exceptions': 1, 'execution_time': 0.0005104541778564453, 'coverage': {}}
Few-shot Line coverage: N/A (Test failed or no coverage data)
Few-shot Branch coverage: N/A (Test failed or no coverage data)


## `numerical_letter_grade` Test generation

In [14]:
from poly_llm.to_test.numerical_letter_grade import numerical_letter_grade

### Few-shot Test Generation with the LLM (`numerical_letter_grade`)

In [15]:

few_shot_examples = ['''def test_numerical_letter_grade(): \n
    assert numerical_letter_grade([4.0, 3, 1.7, 2, 3.5]) == ['A+', 'B', 'C-', 'C', 'A-'] \n
    assert numerical_letter_grade([1.2]) == ['D+'] \n''']
few_numerical_letter_grade_test, few_numerical_letter_grade_test_name = create_test_function(numerical_letter_grade, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_numerical_letter_grade_test_filename = "codegen_few_test_numerical_letter_grade.py"
save_test_to_file(few_numerical_letter_grade_test, numerical_letter_grade, filename=few_numerical_letter_grade_test_filename)

Generate unit tests for the provided input file.
Input
def numerical_letter_grade(grades):
    

   
    letter_grade = []
    for gpa in grades:
        if gpa == 4.0:
            letter_grade.append("A+")
        elif gpa > 3.7:
            letter_grade.append("A")
        elif gpa > 3.3:
            letter_grade.append("A-")
        elif gpa > 3.0:
            letter_grade.append("B+")
        elif gpa > 2.7:
            letter_grade.append("B")
        elif gpa > 2.3:
            letter_grade.append("B-")
        elif gpa > 2.0:
            letter_grade.append("C+")
        elif gpa > 1.7:
            letter_grade.append("C")
        elif gpa > 1.3:
            letter_grade.append("C-")
        elif gpa > 1.0:
            letter_grade.append("D+")
        elif gpa > 0.7:
            letter_grade.append("D")
        elif gpa > 0.0:
            letter_grade.append("D-")
        else:
            letter_grade.append("E")
    return letter_grade

Below is a list of test case examples.


### Generated test coverage (`numerical_letter_grade`)

In [19]:
few_numerical_letter_grade_llm_test_coverage = coverage_llm_produced_code(few_numerical_letter_grade_test_filename, numerical_letter_grade, few_numerical_letter_grade_test_name)
print(few_numerical_letter_grade_llm_test_coverage)

few_line_cov = few_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')


print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

{'input': <function numerical_letter_grade at 0x7925185be160>, 'exceptions': 0, 'execution_time': 0.0002002716064453125, 'coverage': {'covered_lines': 25, 'num_statements': 33, 'percent_covered': 76.27118644067797, 'percent_covered_display': '76', 'missing_lines': 8, 'excluded_lines': 3, 'percent_statements_covered': 75.75757575757575, 'percent_statements_covered_display': '76', 'num_branches': 26, 'num_partial_branches': 6, 'covered_branches': 20, 'missing_branches': 6, 'percent_branches_covered': 76.92307692307692, 'percent_branches_covered_display': '77'}}
Few-shot Line coverage: 76.27118644067797
Few-shot Branch coverage: 76.92307692307692


## `separate_paren_groups` Test generation

In [17]:
from poly_llm.to_test.separate_paren_groups import separate_paren_groups

### Few-shot Test Generation with the LLM (`separate_paren_groups`)

In [22]:

few_shot_examples = ['''def test_separate_paren_groups(): \n
    assert separate_paren_groups('(()()) ((())) () ((())()())') == [ '(()())', '((()))', '()', '((())()())' ] \n
    assert separate_paren_groups('() (()) ((())) (((())))') == [ '()', '(())', '((()))', '(((())))' ] \n''']
few_separate_paren_groups_test, few_separate_paren_groups_test_name = create_test_function(separate_paren_groups, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_separate_paren_groups_test_filename = "codegen_few_test_separate_paren_group.py"
save_test_to_file(few_separate_paren_groups_test, separate_paren_groups, filename=few_separate_paren_groups_test_filename)

Generate unit tests for the provided input file.
Input
def separate_paren_groups(paren_string: str) -> List[str]:
    
    result = []
    current_string = []
    current_depth = 0

    for c in paren_string:
        if c == '(':
            current_depth += 1
            current_string.append(c)
        elif c == ')':
            current_depth -= 1
            current_string.append(c)

            if current_depth == 0:
                result.append(''.join(current_string))
                current_string.clear()

    return result

Below is a list of test case examples.
Example
def test_separate_paren_groups(): 

    assert separate_paren_groups('(()()) ((())) () ((())()())') == [ '(()())', '((()))', '()', '((())()())' ] 

    assert separate_paren_groups('() (()) ((())) (((())))') == [ '()', '(())', '((()))', '(((())))' ] 


Test function written to codegen_few_test_separate_paren_group.py


### Generated test coverage (`separate_paren_groups`)

In [25]:
few_separate_paren_group_llm_test_coverage = coverage_llm_produced_code(few_separate_paren_groups_test_filename, separate_paren_groups, few_separate_paren_groups_test_name)
print(few_separate_paren_group_llm_test_coverage)

few_line_cov = few_separate_paren_group_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_separate_paren_group_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')

print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

{'input': <function separate_paren_groups at 0x7925186658a0>, 'exceptions': 1, 'execution_time': 0.00023293495178222656, 'coverage': {}}
Few-shot Line coverage: N/A (Test failed or no coverage data)
Few-shot Branch coverage: N/A (Test failed or no coverage data)


## `file_name_check` Test generation

In [26]:
from poly_llm.to_test.file_name_check import file_name_check

### Few-shot Test Generation with the LLM

In [27]:
# Creation of the test suite
few_shot_examples = ['''def test_file_name_check(): \n
    assert file_name_check("example.txt") == 'Yes'  \n
    assert file_name_check("1example.dll") == 'No' \n
    assert file_name_check('.txt') == 'No' \n''']
few_file_name_check_test, few_file_name_check_test_name = create_test_function(file_name_check, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_file_name_check_test_filename = "codegen_few_test_file_name_check.py"
save_test_to_file(few_file_name_check_test, file_name_check, filename=few_file_name_check_test_filename)

Generate unit tests for the provided input file.
Input
def file_name_check(file_name):
    
    suf = ['txt', 'exe', 'dll']
    lst = file_name.split(sep='.')
    if len(lst) != 2:
        return 'No'
    if not lst[1] in suf:
        return 'No'
    if len(lst[0]) == 0:
        return 'No'
    if not lst[0][0].isalpha():
        return 'No'
    t = len([x for x in lst[0] if x.isdigit()])
    if t > 3:
        return 'No'
    return 'Yes'

Below is a list of test case examples.
Example
def test_file_name_check(): 

    assert file_name_check("example.txt") == 'Yes'  

    assert file_name_check("1example.dll") == 'No' 

    assert file_name_check('.txt') == 'No' 


Test function written to codegen_few_test_file_name_check.py


### Generated test coverage

In [29]:
few_file_name_check_llm_test_coverage = coverage_llm_produced_code(few_file_name_check_test_filename, file_name_check, few_file_name_check_test_name)
print(few_file_name_check_llm_test_coverage)

print(f"Few-shot Line coverage: {few_file_name_check_llm_test_coverage['coverage']['percent_covered']}")
print(f"Few-shot Branch coverage: {few_file_name_check_llm_test_coverage['coverage']['percent_branches_covered']}")

{'input': <function file_name_check at 0x792518665300>, 'exceptions': 0, 'execution_time': 0.0002079010009765625, 'coverage': {'covered_lines': 16, 'num_statements': 21, 'percent_covered': 74.19354838709677, 'percent_covered_display': '74', 'missing_lines': 5, 'excluded_lines': 4, 'percent_statements_covered': 76.19047619047619, 'percent_statements_covered_display': '76', 'num_branches': 10, 'num_partial_branches': 3, 'covered_branches': 7, 'missing_branches': 3, 'percent_branches_covered': 70.0, 'percent_branches_covered_display': '70'}}
Few-shot Line coverage: 74.19354838709677
Few-shot Branch coverage: 70.0


## `closest_integer` Test generation

In [30]:
from poly_llm.to_test.closest_integer import closest_integer

### Few-shot Test Generation with the LLM

In [31]:
# Creation of the test suite
few_shot_examples = ['''def test_closest_integer(): \n
    assert closest_integer("10") == 10, \n
    assert closest_integer("14.5") == 15 \n''']
few_closest_integer_test, few_closest_integer_test_name = create_test_function(closest_integer, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_closest_integer_test_filename = "codegen_few_test_closest_integer.py"
save_test_to_file(few_closest_integer_test, closest_integer, filename=few_closest_integer_test_filename)

Generate unit tests for the provided input file.
Input
def closest_integer(value):
    '''
    Create a function that takes a value (string) representing a number
    and returns the closest integer to it. If the number is equidistant
    from two integers, round it away from zero.
    Note:
    Rounding away from zero means that if the given number is equidistant
    from two integers, the one you should return is the one that is the
    farthest from zero. For example closest_integer("14.5") should
    return 15 and closest_integer("-14.5") should return -15.
    '''
    from math import floor, ceil

    if value.count('.') == 1:
        # remove trailing zeros
        while (value[-1] == '0'):
            value = value[:-1]

    num = float(value)
    if value[-2:] == '.5':
        if num > 0:
            res = ceil(num)
        else:
            res = floor(num)
    elif len(value) > 0:
        res = int(round(num))
    else:
        res = 0

    return res

Below is a list of test

### Generated test coverage

In [ ]:
few_closest_integer_llm_test_coverage = coverage_llm_produced_code(few_closest_integer_test_filename, closest_integer, few_closest_integer_test_name)
print(few_file_name_check_llm_test_coverage)

print(f"Zero-shot Line coverage: {few_closest_integer_llm_test_coverage['coverage']['percent_covered']}")
print(f"Zero-shot Branch coverage: {few_closest_integer_llm_test_coverage['coverage']['percent_branches_covered']}")